# Python Metaclasses
Covers: type() as metaclass, __new__ vs __init__, custom metaclass, __init_subclass__, class decorators vs metaclasses, Singleton, Registry

## 1. type() — Everything is an Object

In [ ]:
# Classes are instances of type
class Foo:
    x = 42

print('type(Foo):', type(Foo))        # <class 'type'>
print('type(Foo()):', type(Foo()))    # <class '__main__.Foo'>
print('isinstance(Foo, type):', isinstance(Foo, type))  # True
print('type(type):', type(type))      # type is its own metaclass!
print('type(object):', type(object))  # <class 'type'>

# Create a class dynamically with type(name, bases, namespace)
Animal = type('Animal', (object,), {
    'sound': 'generic',
    'speak': lambda self: f'I say {self.sound}',
    '__repr__': lambda self: f'Animal(sound={self.sound!r})'
})

Dog = type('Dog', (Animal,), {
    'sound': 'woof',
})

d = Dog()
print('\nDynamic class:', d)
print('speak():', d.speak())
print('isinstance(d, Animal):', isinstance(d, Animal))
print('Dog.__bases__:', Dog.__bases__)

## 2. Custom Metaclass — __new__ and __init__

In [ ]:
class TracingMeta(type):
    """Metaclass that traces class creation."""
    
    def __new__(mcs, name, bases, namespace):
        print(f'[__new__] Creating class: {name}')
        print(f'  bases: {bases}')
        print(f'  namespace keys: {[k for k in namespace if not k.startswith("__")]}')
        
        # Add metadata to every class
        namespace['_created_by'] = 'TracingMeta'
        namespace['_methods'] = [k for k, v in namespace.items() 
                                   if callable(v) and not k.startswith('_')]
        
        cls = super().__new__(mcs, name, bases, namespace)
        return cls
    
    def __init__(cls, name, bases, namespace):
        print(f'[__init__] Initializing class: {name}')
        super().__init__(name, bases, namespace)

class MyService(metaclass=TracingMeta):
    version = '1.0'
    
    def process(self, data):
        return data
    
    def validate(self, data):
        return bool(data)

print('\nMyService._created_by:', MyService._created_by)
print('MyService._methods:', MyService._methods)

## 3. __init_subclass__ — Simpler Alternative

In [ ]:
class Plugin:
    """Base class with automatic plugin registration."""
    _registry = {}
    
    def __init_subclass__(cls, plugin_name=None, version='1.0', **kwargs):
        super().__init_subclass__(**kwargs)
        name = plugin_name or cls.__name__.lower()
        Plugin._registry[name] = cls
        cls._plugin_name = name
        cls._plugin_version = version
        print(f'Registered: {name} v{version} -> {cls.__name__}')
    
    @classmethod
    def get(cls, name):
        if name not in cls._registry:
            raise KeyError(f'Plugin not found: {name}')
        return cls._registry[name]()

class JSONPlugin(Plugin, plugin_name='json', version='2.0'):
    def process(self, data):
        import json
        return json.dumps(data)

class CSVPlugin(Plugin, plugin_name='csv'):
    def process(self, data):
        return ','.join(str(x) for x in data)

class XMLPlugin(Plugin):  # uses class name
    def process(self, data):
        return f'<data>{data}</data>'

print('\nRegistry:', list(Plugin._registry.keys()))

# Use plugins
json_plugin = Plugin.get('json')
print('JSON:', json_plugin.process({'key': 'value'}))

csv_plugin = Plugin.get('csv')
print('CSV:', csv_plugin.process([1, 2, 3, 4, 5]))

## 4. Singleton Pattern with Metaclass

In [ ]:
import threading

class SingletonMeta(type):
    """Thread-safe Singleton metaclass."""
    _instances = {}
    _lock = threading.Lock()
    
    def __call__(cls, *args, **kwargs):
        # Double-checked locking
        if cls not in cls._instances:
            with cls._lock:
                if cls not in cls._instances:
                    instance = super().__call__(*args, **kwargs)
                    cls._instances[cls] = instance
        return cls._instances[cls]

class Config(metaclass=SingletonMeta):
    def __init__(self, env='production'):
        self.env = env
        self.settings = {}
        print(f'Config created for env: {env}')
    
    def set(self, key, value):
        self.settings[key] = value
    
    def get(self, key, default=None):
        return self.settings.get(key, default)

# Only one instance created
c1 = Config('development')
c2 = Config('production')  # __init__ NOT called again
c3 = Config()

print('c1 is c2:', c1 is c2)  # True
print('c1 is c3:', c1 is c3)  # True

c1.set('debug', True)
print('c2.get(debug):', c2.get('debug'))  # True — same instance!

# Thread safety test
instances = []
def create_config():
    instances.append(Config())

threads = [threading.Thread(target=create_config) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()

print(f'\nAll {len(instances)} instances are same: {len(set(id(i) for i in instances)) == 1}')

## 5. Class Decorators vs Metaclasses

In [ ]:
import functools
import time

# Class decorator — simpler, for single class modification
def auto_repr(cls):
    """Add __repr__ based on __init__ parameters."""
    @functools.wraps(cls.__init__)
    def __repr__(self):
        attrs = ', '.join(f'{k}={v!r}' for k, v in self.__dict__.items())
        return f'{cls.__name__}({attrs})'
    cls.__repr__ = __repr__
    return cls

def validate_types(**type_map):
    """Class decorator that validates __init__ argument types."""
    def decorator(cls):
        original_init = cls.__init__
        @functools.wraps(original_init)
        def new_init(self, **kwargs):
            for param, expected_type in type_map.items():
                if param in kwargs and not isinstance(kwargs[param], expected_type):
                    raise TypeError(
                        f'{param} must be {expected_type.__name__}, '
                        f'got {type(kwargs[param]).__name__}'
                    )
            original_init(self, **kwargs)
        cls.__init__ = new_init
        return cls
    return decorator

@auto_repr
@validate_types(name=str, age=int)
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person(name='Alice', age=30)
print('Person:', p)

try:
    bad = Person(name='Bob', age='thirty')  # wrong type
except TypeError as e:
    print('TypeError:', e)

# Metaclass — affects ALL subclasses automatically
class AutoReprMeta(type):
    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)
        def __repr__(self):
            attrs = ', '.join(f'{k}={v!r}' for k, v in self.__dict__.items())
            return f'{name}({attrs})'
        cls.__repr__ = __repr__
        return cls

class Base(metaclass=AutoReprMeta):
    pass

class Point(Base):  # automatically gets __repr__!
    def __init__(self, x, y):
        self.x = x
        self.y = y

class Color(Base):  # also gets __repr__!
    def __init__(self, r, g, b):
        self.r = r; self.g = g; self.b = b

print('\nPoint:', Point(3, 4))
print('Color:', Color(255, 128, 0))